# D6 Sentence-Pair Swap — No Shuffles

Raw marginals only. ~15 min per corpus.
Compares against existing intact + D4 corrected marginals from formal run.

In [ ]:
!pip install -q -U bitsandbytes>=0.46.1 accelerate

import numpy as np
import json, math, os, gc, random, time, re, shutil
from pathlib import Path
from collections import Counter
from scipy import stats
from scipy.ndimage import uniform_filter1d
from tqdm.auto import tqdm
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

from google.colab import drive
drive.mount('/content/drive')

BASE = Path('/content/drive/MyDrive/LRTIA/Results/Exp1A_D6')
BASE.mkdir(parents=True, exist_ok=True)
FORMAL = Path('/content/drive/MyDrive/LRTIA/Results/Exp1A_formal')
DATA = Path('/content/drive/MyDrive/LRTIA/Data')

CORPORA = {
    'wiki_zh': DATA / 'wiki_multilingual/zh_articles.jsonl',
    'wiki_ja': DATA / 'wiki_multilingual/ja_articles.jsonl',
}

MODEL_NAME = 'unsloth/Meta-Llama-3.1-8B'
C = 100
TARGET_LEN = 30
TARGET_FRACS = [0.25, 0.50, 0.75]
MIN_BEFORE = C + 10
N_SHUFFLES = 1
SEED = 20260429

# Copy existing intact + D4 results
for cn in CORPORA:
    for cond in ['intact', 'D4']:
        src = FORMAL / f'llama_{cn}_{cond}.json'
        dst = BASE / f'llama_{cn}_{cond}.json'
        if src.exists() and not dst.exists():
            shutil.copy2(src, dst)
            print(f'Copied {cn} {cond}')

print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'{N_SHUFFLES} shuffle — ~30 min total')
print('Setup done')

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.float16),
    device_map='auto'
)
model.eval()
print('Llama loaded')

In [ ]:
# === D6 + PPL functions (NO SHUFFLES) ===

SENT_BOUNDS = set('. ? ! 。 ？ ！ ؟'.split())

def segment_sentences(token_ids, tokenizer):
    units = []
    start = 0
    for i, tid in enumerate(token_ids):
        s = tokenizer.decode([tid]).strip()
        if any(ch in SENT_BOUNDS for ch in s):
            units.append((start, i + 1))
            start = i + 1
    if start < len(token_ids):
        units.append((start, len(token_ids)))
    return units

def d6_swap(ctx, tokenizer):
    units = segment_sentences(ctx, tokenizer)
    if len(units) < 4:
        return None, len(units), False
    swapped = []
    i = 0
    while i < len(units) - 1:
        swapped.append(units[i + 1])
        swapped.append(units[i])
        i += 2
    if i < len(units):
        swapped.append(units[i])
    result = []
    for s, e in swapped:
        result.extend(ctx[s:e])
    return result, len(units), True

@torch.no_grad()
def ppl_only(ctx_toks, tgt_toks):
    if len(tgt_toks) < 2: return float('inf')
    full = list(ctx_toks) + list(tgt_toks)
    ts = len(ctx_toks)
    ids = torch.tensor([full], device=model.device)
    out = model(ids)
    logits = out.logits[0]
    nll = 0.0; cnt = 0
    for i in range(ts, len(full) - 1):
        lp = torch.log_softmax(logits[i], dim=-1)
        nll += -lp[full[i+1]].item(); cnt += 1
    del out, logits; torch.cuda.empty_cache()
    if cnt == 0: return float('inf')
    return math.exp(nll / cnt)

def compute_raw_curves(cond_ctx, tgt):
    """Raw marginals — no shuffles. 101 forward passes."""
    mc = len(cond_ctx)
    ppls = []
    for c in range(mc + 1):
        pfx = cond_ctx[-c:] if c > 0 else []
        ppls.append(ppl_only(pfx, tgt))
    dists = list(range(1, mc + 1))
    marginals = [ppls[d-1] - ppls[d] for d in dists]
    return {'distances': dists, 'ppls': ppls, 'marginals': marginals}

# Quick test
with open(CORPORA['wiki_zh']) as f:
    doc = json.loads(f.readline())
ids = tokenizer.encode(doc['text'], add_special_tokens=False)
ctx = ids[len(ids)//2 - C : len(ids)//2]
d6, nu, ok = d6_swap(ctx, tokenizer)
print(f'Test: {nu} units, eligible={ok}, len={len(d6) if d6 else 0}, multiset ok={Counter(d6)==Counter(ctx) if d6 else False}')
print('Functions ready — NO SHUFFLES')

In [ ]:
# === Run D6 + intact (raw, no shuffles) ===

d6_stats = {}

for cn, cp in CORPORA.items():
    print(f'\n{"="*60}')
    print(cn)
    print(f'{"="*60}')
    
    docs = []
    with open(cp) as f:
        for line in f: docs.append(json.loads(line))
    
    for cond_name in ['intact', 'D6']:
        cache = BASE / f'llama_{cn}_{cond_name}_raw.json'
        if cache.exists():
            with open(cache) as f: n = len(json.load(f))
            print(f'  {cond_name}: cached ({n})'); continue
        
        t0 = time.time()
        results = []
        n_elig, n_tot = 0, 0
        all_units = []
        
        for doc in tqdm(docs, desc=f'{cn}/{cond_name}'):
            fids = tokenizer.encode(doc['text'], add_special_tokens=False)
            n = len(fids)
            for frac in TARGET_FRACS:
                ts = int(n * frac)
                te = min(ts + TARGET_LEN, n)
                if ts < MIN_BEFORE or te - ts < 5: continue
                ctx = fids[ts-C:ts]
                tgt = fids[ts:te]
                n_tot += 1
                
                if cond_name == 'intact':
                    cond_ctx = list(ctx)
                else:
                    cond_ctx, nu, ok = d6_swap(ctx, tokenizer)
                    all_units.append(nu)
                    if not ok: continue
                    n_elig += 1
                    assert len(cond_ctx) == C and Counter(cond_ctx) == Counter(ctx)
                
                r = compute_raw_curves(cond_ctx, tgt)
                r['doc_id'] = doc.get('doc_id', '')
                r['target_frac'] = frac
                if cond_name == 'D6': r['n_units'] = nu
                results.append(r)
        
        with open(cache, 'w') as f: json.dump(results, f)
        elapsed = time.time() - t0
        print(f'  {cond_name}: {len(results)} results in {elapsed/60:.1f} min')
        
        if cond_name == 'D6':
            elig_rate = n_elig / n_tot if n_tot > 0 else 0
            d6_stats[cn] = {'elig': elig_rate, 'units': np.mean(all_units), 'n': n_elig, 'total': n_tot}
            print(f'  Eligible: {n_elig}/{n_tot} ({elig_rate:.0%}), mean units: {np.mean(all_units):.1f}')
        
        if results:
            md = np.mean([np.mean(r['marginals']) for r in results])
            print(f'  Mean raw marginal: {md:.6f}')

In [ ]:
# === Results: raw marginals for intact vs D6, plus existing D4 ===
import matplotlib.pyplot as plt

print(f'{"Corpus":<12} {"Cond":<10} {"TotalΔ":>10} {"NearΔ":>10} {"MidΔ":>10} {"FarΔ":>10}')
print(f'{"":>12} {"":>10} {"d=1..100":>10} {"d=1..30":>10} {"d=31..70":>10} {"d=71..100":>10}')
print('-' * 65)

for cn in CORPORA:
    for cond, src_base, fname, key in [
        ('intact', BASE, f'llama_{cn}_intact_raw.json', 'marginals'),
        ('D6',     BASE, f'llama_{cn}_D6_raw.json', 'marginals'),
        ('D4(corr)', FORMAL, f'llama_{cn}_D4.json', 'delta_ppl'),
    ]:
        cp = src_base / fname
        if not cp.exists():
            print(f'{cn:<12} {cond:<10} {"—":>10}'); continue
        with open(cp) as f: results = json.load(f)
        if not results: continue
        curve = np.mean([r[key] for r in results], axis=0)
        total = np.mean(curve)
        near = np.mean(curve[:30])
        mid = np.mean(curve[30:70])
        far = np.mean(curve[70:])
        print(f'{cn:<12} {cond:<10} {total:>10.4f} {near:>10.4f} {mid:>10.4f} {far:>10.4f}')
    print()

if d6_stats:
    print('D6 eligibility:')
    for cn, st in d6_stats.items():
        print(f'  {cn}: {st["n"]}/{st["total"]} ({st["elig"]:.0%}), {st["units"]:.1f} units')

# Plot: PPL curves (easier to interpret than marginals)
fig, axes = plt.subplots(1, len(CORPORA), figsize=(7*len(CORPORA), 5))
if len(CORPORA) == 1: axes = [axes]

for idx, cn in enumerate(CORPORA):
    ax = axes[idx]
    for cond, color in [('intact', 'blue'), ('D6', 'orange')]:
        cp = BASE / f'llama_{cn}_{cond}_raw.json'
        if not cp.exists(): continue
        with open(cp) as f: r = json.load(f)
        if not r: continue
        ppl_curve = np.mean([x['ppls'] for x in r], axis=0)
        ax.plot(range(len(ppl_curve)), ppl_curve, color=color, linewidth=2, label=cond)
    ax.axhline(0, color='gray', linestyle=':', alpha=0.3)
    ax.set_title(cn, fontweight='bold')
    ax.set_xlabel('Context length c')
    ax.set_ylabel('Perplexity')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.15)

plt.suptitle('D6 (sentence swap) vs Intact — PPL Curves',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(BASE / 'fig_D6_ppl.png', dpi=150, bbox_inches='tight')
plt.show()

# Also raw marginals
fig, axes = plt.subplots(1, len(CORPORA), figsize=(7*len(CORPORA), 5))
if len(CORPORA) == 1: axes = [axes]

for idx, cn in enumerate(CORPORA):
    ax = axes[idx]
    for cond, color in [('intact', 'blue'), ('D6', 'orange')]:
        cp = BASE / f'llama_{cn}_{cond}_raw.json'
        if not cp.exists(): continue
        with open(cp) as f: r = json.load(f)
        if not r: continue
        curve = np.mean([x['marginals'] for x in r], axis=0)
        ax.plot(range(1, len(curve)+1), uniform_filter1d(curve, 5),
                color=color, linewidth=2, label=cond)
    ax.axhline(0, color='gray', linestyle=':', alpha=0.3)
    ax.set_title(cn, fontweight='bold')
    ax.set_xlabel('Distance d')
    ax.set_ylabel('Raw Marginal')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.15)

plt.suptitle('D6 (sentence swap) vs Intact — Raw Marginals',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(BASE / 'fig_D6_marginals.png', dpi=150, bbox_inches='tight')
plt.show()
print('Done')